In [14]:
import pandas as pd
import yfinance as yf
import financedatabase as fd

import importlib
import utils.price_increase as pi

importlib.reload(pi)

import requests
from bs4 import BeautifulSoup

import time
import random
import re

# from utils.price_increase import yearly_price_increase, multi_year_performance

In [15]:
# Get fund ticker data
fundsTicker = pd.read_csv("data/funds_ticker.csv")
euroFunds = fundsTicker[fundsTicker['currency'] == "EUR"]
euroFunds = euroFunds[["name", "symbol", "currency"]]
print(euroFunds.shape)
def findTicker(fundName, partialTicker):
    phrase = fundName[:20]

    matches = euroFunds[
        # (euroFunds["name"].str.contains(phrase, case=False, na=False)) 
                        #   & 
                          (euroFunds["symbol"].str.contains(partialTicker, case=False, na=False))]
    if matches.shape[0] > 1:
        m2 = matches[matches["symbol"].str.contains(rf"{partialTicker}\.", case=False, na=False)]
        if m2.shape[0] > 0:
            return m2
    return matches

tt = findTicker("BNP Paribas Easy MSCI Pacific ex Japan Min TE UCITS ETF", "PAC")
tt

(27875, 3)


,name,symbol,currency
23438,BNP Paribas Easy MSCI Pacific ex Japan ex CW U...,PAC.DE,EUR
23439,BNPPE-MSCI PA.XJXCW.UECEO,PAC.DU,EUR
23440,BNP Paribas Easy MSCI Pacific ex Japan ex CW U...,PAC.F,EUR


In [22]:
etfData = pd.read_excel("data/etfs.xlsx")

# onlyEtf = etfData.where(etfData["assetClass"] == "equity").dropna()
onlyEtf = etfData[(etfData["asset_class"] == "equity") 
                #   & (etfData["currency"] == "EUR") 
                  & (etfData["dividends"] == "Accumulating") & (etfData["number_of_holdings"] > 50)]

onlyEtf = onlyEtf[["isin", "name", "ticker","size", "asset_class", "currency", "domicile_country", "hedged", "dividends", "ter", "number_of_holdings", "index"]]
print(onlyEtf.shape)
print(etfData.shape)

for index, row in onlyEtf.iterrows():
    try:
        possibleTickers = findTicker(row['name'], row['ticker'])
        if not possibleTickers.empty:
            # row["symbol"] = possibleTickers.iloc[0]['symbol']
            onlyEtf.loc[index, "symbol"] = possibleTickers.iloc[0]['symbol']
            print(f"Match found for {row['name']} with ticker {row['ticker']}: {row['symbol']}")
        # else:
        #     print(f"No match found for {row['name']} with ticker {row['ticker']}")
    except Exception as e:
        # print(f"Error processing {row['ticker']}: {e}")
        continue


onlyEtf.to_csv("etf-with-ticker.csv", index=False)
onlyEtf.head()

(958, 12)
(4062, 21)


,isin,name,ticker,size,asset_class,currency,domicile_country,hedged,dividends,ter,number_of_holdings,index,symbol
131,DE000A2QP4B6,iShares STOXX Europe 600 UCITS ETF (DE) EUR (Acc),EXIE,849.0,equity,EUR,Germany,False,Accumulating,0.20,600.0,STOXX® Europe 600,NaN
132,DE000A2QP4C4,iShares Dow Jones Global Titans 50 UCITS ETF (...,EXIF,4.0,equity,EUR,Germany,False,Accumulating,0.51,53.0,Dow Jones Global Titans 50,NaN
268,FR0007054358,Amundi EURO STOXX 50 II UCITS ETF Acc,LYSX,3923.0,equity,EUR,France,False,Accumulating,0.20,54.0,EURO STOXX® 50,LYSX.BE
275,FR0010261198,Amundi MSCI Europe UCITS ETF Acc,LYY5,611.0,equity,EUR,France,False,Accumulating,0.25,406.0,MSCI Europe,LYY5.BE
294,FR0010655704,Amundi CAC Transition Climat UCITS ETF,X13J,11.0,equity,EUR,France,False,Accumulating,0.25,69.0,CAC Transition Climat,X13J.BE


In [17]:
# Fetch historical data from yahoo finance for a given ticker
def fetchHistoricData(ticker):
    etf = yf.Ticker(ticker)

    current_year = pd.Timestamp.today().year
    start_year = current_year - 15   # inclusive = 10 years

    start = pd.Timestamp(f"{start_year}-01-01")
    end = pd.Timestamp(f"{current_year - 1}-12-31")

    hist = etf.history(start=start, end=end)
    if hist.empty:
        return None


    yearly = pi.yearly_price_increase(hist)

    tt =  yearly[['pct_increase']].T
    return tt
    perf = multi_year_performance(yearly, periods=(3,5,10))

    oneRow = perf.set_index('Years Ago')[['% Increase']].T.rename(columns={3: '3year', 5: '5year', 10: '10year'})
    # oneRow.index = [0] 
    # result = pd.DataFrame(oneRow.values, columns=oneRow.columns)
    return oneRow

In [53]:
data = fetchHistoricData("PAC.DE")
data 


Not enough history to calculate 10-year performance.


/home/nerjok/Documents/fintech/utils/price_increase.py:66: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  nf = pd.concat([pd.DataFrame([new_row]), nf], ignore_index=True)


Year,5 year,3 year,1 year,2016-12-31 00:00:00+01:00,2017-12-31 00:00:00+01:00,2018-12-31 00:00:00+01:00,2019-12-31 00:00:00+01:00,2020-12-31 00:00:00+01:00,2021-12-31 00:00:00+01:00,2022-12-31 00:00:00+01:00,2023-12-31 00:00:00+01:00,2024-12-31 00:00:00+01:00,2025-12-31 00:00:00+01:00
pct_increase,38.879824,22.454854,6.727961,6.302722,7.004185,-95.721293,21.448477,-2.715567,12.83073,0.903076,1.489232,12.136832,5.392025


In [51]:
# from bs4 import soup


def justetf_perf(isin):
    url = f"https://www.justetf.com/en/etf-profile.html?isin={isin}"
    headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.justetf.com/"
    }
    html = requests.get(url, headers=headers).text
    soup = BeautifulSoup(html, "html.parser")
    
    perf = {}
    # rows = soup.select("table tbody tr")

    # Add volatility
    volatilityTable = soup.find("table", {"data-testid": "etf-basics_data_table"})
    rows2 = volatilityTable.find_all("tr")
    for r in rows2:
        cols2 = [c.text.strip() for c in r.find_all("td")]

        if len(cols2) != 2:
            continue
        if "Volatility" in cols2[0]:
            perf["Volatility"] = cols2[1]
            continue
        if len(cols2) == 2:
            perf[cols2[0]] = cols2[1]


    # Add performance
    table = soup.find("table", {"data-testid": "etf-returns-section_table"})
    if not table:
        return None

    data = {}

    rows = table.find_all("tr")
    # print(rows)
    for r in rows:
        cols = [c.text.strip() for c in r.find_all("td")]
        if len(cols) == 2:
            perf[cols[0]] = cols[1]
    # print(perf)
    perf_filtered = {k.rstrip('s'): v for k, v in perf.items() 
                     if k in ["1 year", "3 years", "5 years", "Volatility", "Fund size"]}
    df = pd.DataFrame(perf_filtered, index=[0])
    return df

print("Results for DE0005933931:")
print(justetf_perf("DE0005933931"))

Results for DE0005933931:
     Fund size Volatility   1 year   3 year   5 year
0  EUR 8,500 m     15.54%  +15.83%  +53.70%  +55.29%


In [55]:
#  Fetch historical data from etffDarabase list that calls justetf for each etf and combines the data into a single dataframe
# onlyEtf = onlyEtf.head(50)  # limit to 10 for testing
print(f"onlyEtf shape: {onlyEtf.shape}")
print(f"onlyEtf empty: {onlyEtf.empty}")
withHistory = pd.DataFrame()
print(onlyEtf.head(10))

def safe_sleep(base=2):
    jitter = random.uniform(0, 2)
    time.sleep(base + jitter)


i = 0
for index, row in onlyEtf.iterrows():
    # ticker = row["symbol"] if not pd.isna(row["symbol"]) else row["isin"]
    ticker = row["symbol"] if not pd.isna(row["symbol"]) else row["isin"]
    # i += 1
    # if i >100:
    #     break
    # print(f"Fetching data for {ticker}...")
    safe_sleep()  # be polite to the server
    try:


        perf = justetf_perf(row["isin"])

        if perf is None:
            perf = fetchHistoricData(ticker)
        if perf is not None:
            perf_dict = perf.iloc[0].to_dict()
            combined = {**row.to_dict(), **perf_dict}
            withHistory = pd.concat([withHistory, pd.DataFrame([combined])], ignore_index=True)
    except Exception as e:
        print(f"Error fetching data for {ticker}: {e}")
        continue


withHistory.to_csv("output/justetf_hist.csv", index=False)
withHistory



onlyEtf shape: (958, 13)
onlyEtf empty: False
             isin                                               name  ticker  \
131  DE000A2QP4B6  iShares STOXX Europe 600 UCITS ETF (DE) EUR (Acc)    EXIE   
132  DE000A2QP4C4  iShares Dow Jones Global Titans 50 UCITS ETF (...    EXIF   
268  FR0007054358              Amundi EURO STOXX 50 II UCITS ETF Acc    LYSX   
275  FR0010261198                   Amundi MSCI Europe UCITS ETF Acc    LYY5   
294  FR0010655704             Amundi CAC Transition Climat UCITS ETF    X13J   
302  FR0010821819  Amundi ETF MSCI Europe Ex EMU ESG Leaders UCIT...    540H   
317  FR0011720911                  Amundi MSCI China A UCITS ETF Acc    CNAA   
318  FR0011758085      Amundi FTSE Italia PMI PIR 2020 UCITS ETF Acc  ITAMID   
320  FR0012399731   Amundi EURO STOXX 50 II UCITS ETF CHF Hedged Acc    MSEC   
321  FR0012399772   Amundi EURO STOXX 50 II UCITS ETF GBP Hedged Acc    NK4B   

       size asset_class currency domicile_country  hedged     dividends  

,isin,name,ticker,size,asset_class,currency,domicile_country,hedged,dividends,ter,number_of_holdings,index,symbol,Fund size,Volatility,1 year,3 year,5 year
0,DE000A2QP4B6,iShares STOXX Europe 600 UCITS ETF (DE) EUR (Acc),EXIE,849.0,equity,EUR,Germany,False,Accumulating,0.20,600.0,STOXX® Europe 600,NaN,EUR 846 m,11.62%,+27.01%,+46.26%,-
1,DE000A2QP4C4,iShares Dow Jones Global Titans 50 UCITS ETF (...,EXIF,4.0,equity,EUR,Germany,False,Accumulating,0.51,53.0,Dow Jones Global Titans 50,NaN,EUR 7 m,-,-,-,-
2,FR0007054358,Amundi EURO STOXX 50 II UCITS ETF Acc,LYSX,3923.0,equity,EUR,France,False,Accumulating,0.20,54.0,EURO STOXX® 50,LYSX.BE,"EUR 4,186 m",14.99%,+26.19%,+51.15%,+72.76%
3,FR0010261198,Amundi MSCI Europe UCITS ETF Acc,LYY5,611.0,equity,EUR,France,False,Accumulating,0.25,406.0,MSCI Europe,LYY5.BE,EUR 628 m,11.57%,+26.39%,+44.42%,+62.66%
4,FR0010655704,Amundi CAC Transition Climat UCITS ETF,X13J,11.0,equity,EUR,France,False,Accumulating,0.25,69.0,CAC Transition Climat,X13J.BE,EUR 11 m,14.51%,+7.78%,+10.39%,+35.69%
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
952,LU2993390843,BNP Paribas Easy S&P 500 II UCITS ETF EUR Hedg...,EDEA,61.0,equity,EUR,Luxembourg,True,Accumulating,0.08,503.0,S&P 500® (EUR Hedged),NaN,EUR 64 m,-,-,-,-
953,LU3086388124,Amundi MSCI Europe Screened UCITS ETF (Acc),EUSC,389.0,equity,EUR,Luxembourg,False,Accumulating,0.12,375.0,MSCI Europe Screened Select ex Thermal Coal,NaN,EUR 551 m,-,-,-,-
954,LU3180074463,Amundi European Strategic Autonomy UCITS ETF Acc,INDEP,6.0,equity,EUR,Luxembourg,False,Accumulating,0.40,221.0,Euronext European Strategic Autonomy,NaN,EUR 14 m,-,-,-,-
955,XS2399364152,Leverage Shares 5x Long Nasdaq 100 ETP,QQQ5,28.0,equity,USD,Ireland,False,Accumulating,0.75,101.0,Nasdaq 100® Leverage (5x),NaN,EUR 30 m,83.16%,+254.35%,+340.54%,-


In [ ]:
# Fetch bonds data
def getContinent(country):
    df = pd.read_csv("data/countries_by_continents.csv")

    row = df[df["Country"].str.lower() == country.lower()]
    if not row.empty:
        # Return the continent value
        return row.iloc[0]["Continent"]
    else:
        return None

def fetch_bonds():
    url = f"https://www.investing.com/rates-bonds/world-government-bonds"
    headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.justetf.com/"
    }
    html = requests.get(url, headers=headers).text
    soup = BeautifulSoup(html, "html.parser")


    tablesContainer = soup.find("section", {"id": "leftColumn"})
    tables = tablesContainer.find_all("table" )
    if not tables:
        return None

    rows_data = []
    for table in tables:
        # if table.find_parent("div", class_="leftColumn") is None:
        #     continue
        if "displayNone" in table.get("class", []):
            continue

        rows = table.find_all("tr")
        for r in rows:
            cols = [c.text.strip() for c in r.find_all("td")]
                
            # keep only the first two columns (pad with None if fewer than 2)
            first_two = cols[1:3]
            # Convert yield to float safely
            # try:
            #     yield_value = float(first_two[1])
            # except ValueError:
            #     continue
            
            # # Skip yields below 3
            # if yield_value < 3:
            #     continue

            if len(first_two) < 2 or not first_two[1]:
                continue
            country = re.sub(r" \d+Y| \d+M", "",first_two[0])
            continent = getContinent(country)

            if continent == "Africa" and country != "Egipt":
                continue


            # rows_data.append([country, first_two[1]])
            rows_data.append([first_two[0], continent, first_two[1]])
        

    rows_data.sort(key=lambda x: float(x[2]) if x[1] else 0, reverse=True)

    df = pd.DataFrame(rows_data[1:])

    df.columns = ["Country", "Continent", "Yield"]
    # df = (
    # df.groupby("Country")["Yield"]
    #   .apply(lambda x: ",".join(sorted(map(str, x), reverse=True)))
    #   .reset_index()
    # )
    df["Yield"] = pd.to_numeric(df["Yield"], errors="coerce")
    df = df[df["Yield"] > 4]
    df.to_csv("output/bonds.csv", index=False)

    print(df)
    return rows_data

fetch_bonds()

       Country Continent   Yield
0    Turkey 9M    Europe  36.308
1    Turkey 6M    Europe  36.041
2    Turkey 3Y    Europe  35.960
3    Turkey 3M    Europe  35.417
4    Turkey 5Y    Europe  34.830
..         ...       ...     ...
633   U.K. 40Y      None   5.300
634   U.K. 50Y      None   4.878
645   U.S. 10Y      None   4.132
646   U.S. 20Y      None   4.727
647   U.S. 30Y      None   4.756

[269 rows x 3 columns]


[['Turkey 2Y', 'Europe', '38.770'],
 ['Turkey 9M', 'Europe', '36.308'],
 ['Turkey 6M', 'Europe', '36.041'],
 ['Turkey 3Y', 'Europe', '35.960'],
 ['Turkey 3M', 'Europe', '35.417'],
 ['Turkey 5Y', 'Europe', '34.830'],
 ['Turkey 10Y', 'Europe', '31.275'],
 ['Ukraine 1Y', 'Europe', '24.750'],
 ['Ukraine 2Y', 'Europe', '23.100'],
 ['Ukraine 3Y', 'Europe', '22.150'],
 ['Kazakhstan 6M', 'Asia', '16.650'],
 ['Kazakhstan 1Y', 'Asia', '16.164'],
 ['Kazakhstan 2Y', 'Asia', '16.036'],
 ['Kazakhstan 3Y', 'Asia', '15.700'],
 ['Kazakhstan 4Y', 'Asia', '15.434'],
 ['Kazakhstan 5Y', 'Asia', '15.300'],
 ['Kazakhstan 6Y', 'Asia', '15.171'],
 ['Kazakhstan 7Y', 'Asia', '15.089'],
 ['Kazakhstan 15Y', 'Asia', '15.028'],
 ['Kazakhstan 8Y', 'Asia', '15.017'],
 ['Kazakhstan 10Y', 'Asia', '15.011'],
 ['Kazakhstan 9Y', 'Asia', '15.004'],
 ['Russia 5Y', 'Asia', '14.755'],
 ['Russia 7Y', 'Asia', '14.745'],
 ['Russia 10Y', 'Asia', '14.595'],
 ['Russia 3Y', 'Asia', '14.579'],
 ['Brazil 3M', 'South America', '14.578']

In [29]:
df = pd.read_csv("data/countries_by_continents.csv")

# df = pd.read_excel(xlsx, "Sheet1")

df

country = df[df["Country"] == "Germany"]
country 



,Continent,Country
117,Europe,Germany


In [63]:
# Yearl ygrowth to decimal

df = pd.read_csv("output/justetf_hist.csv")

def parse_percent(x):
    if pd.isna(x) or x == '-':
        return None
    return float(x.replace('%','').replace('+','')) / 100

df['return_1y'] = df['1 year'].apply(parse_percent)
df['return_3y'] = df['3 year'].apply(parse_percent)
df['return_5y'] = df['5 year'].apply(parse_percent)
df['volatility'] = df['Volatility'].apply(parse_percent)

df = df.dropna(subset=['volatility'])

#  different set of time period
df['expected_return'] = df['return_3y']

# Sharpe ratio with 0% risk free rate
df['annual_return'] = (1 + df['return_3y']) ** (1/3) - 1
df['score'] = df['annual_return'] / df['volatility']

df.head()

,isin,name,ticker,size,asset_class,currency,domicile_country,hedged,dividends,ter,...,1 year,3 year,5 year,return_1y,return_3y,return_5y,volatility,expected_return,annual_return,score
0,DE000A2QP4B6,iShares STOXX Europe 600 UCITS ETF (DE) EUR (Acc),EXIE,849.0,equity,EUR,Germany,False,Accumulating,0.20,...,+27.01%,+46.26%,-,0.2701,0.4626,NaN,0.1162,0.4626,0.135120,1.162825
2,FR0007054358,Amundi EURO STOXX 50 II UCITS ETF Acc,LYSX,3923.0,equity,EUR,France,False,Accumulating,0.20,...,+26.19%,+51.15%,+72.76%,0.2619,0.5115,0.7276,0.1499,0.5115,0.147632,0.984871
3,FR0010261198,Amundi MSCI Europe UCITS ETF Acc,LYY5,611.0,equity,EUR,France,False,Accumulating,0.25,...,+26.39%,+44.42%,+62.66%,0.2639,0.4442,0.6266,0.1157,0.4442,0.130340,1.126535
4,FR0010655704,Amundi CAC Transition Climat UCITS ETF,X13J,11.0,equity,EUR,France,False,Accumulating,0.25,...,+7.78%,+10.39%,+35.69%,0.0778,0.1039,0.3569,0.1451,0.1039,0.033499,0.230866
5,FR0010821819,Amundi ETF MSCI Europe Ex EMU ESG Leaders UCIT...,540H,230.0,equity,EUR,France,False,Accumulating,0.30,...,+15.71%,+26.67%,+47.27%,0.1571,0.2667,0.4727,0.1165,0.2667,0.081993,0.703806


In [72]:
# Return etfs with best risk adjusted return (score) for each region and risk level
def parse_percent(x):
    if pd.isna(x) or x == '-':
        return None
    return float(x.replace('%','').replace('+','')) / 100


def build_portfolio(user, etf_df, bonds_df):
    
    # --- 1. PREPARE ETF DATA ---
    etf_df = etf_df.copy()
    
    etf_df['return_3y'] = etf_df['3 year'].apply(parse_percent)
    etf_df['volatility'] = etf_df['Volatility'].apply(parse_percent)

    # Remove rows with missing return or volatility
    etf_df = etf_df.dropna(subset=['return_3y', 'volatility'])

    # Annualize return
    etf_df['annual_return'] = (1 + etf_df['return_3y']) ** (1/3) - 1

    # Score (risk-adjusted) Sharp ratio with 0% risk free rate
    etf_df['score'] = etf_df['annual_return'] / etf_df['volatility']


    # --- 2. FILTER BY USER INPUT ---

    # Risk filter
    def risk_filter(df):
        if user["riskLevel"] == "low":
            return df[df['volatility'] < 0.10]
        elif user["riskLevel"] == "medium":
            return df[(df['volatility'] >= 0.10) & (df['volatility'] < 0.20)]
        else:
            return df[df['volatility'] >= 0.20]

    etf_df = risk_filter(etf_df)

    # Region filter, TODO add where securities are listed, not only domicile
    if not user["selectedRegions"].get("global"):
        if user["selectedRegions"].get("us"):
            etf_df = etf_df[etf_df['domicile_country'] == "USA"]


    # --- 3. PICK BEST ETFs ---
    etf_df = etf_df.sort_values(by='score', ascending=False).head(15)


    # --- 4. CALCULATE TOTAL MONEY ---
    years = int(user["years"].replace("y", ""))
    total_amount = user["perYearInvestment"] * years


    # --- 5. SPLIT ETF / BONDS ---
    risk_split = {
        "low":   {"etf": 0.3, "bonds": 0.7},
        "medium":{"etf": 0.6, "bonds": 0.4},
        "high":  {"etf": 0.9, "bonds": 0.1}
    }

    split = risk_split[user["riskLevel"]]

    etf_budget = total_amount * split["etf"]
    bonds_budget = total_amount * split["bonds"]


    # --- 6. ALLOCATE ETF MONEY ---
    etf_df['weight'] = etf_df['score'] / etf_df['score'].sum()
    etf_df['allocation'] = etf_df['weight'] * etf_budget


    # --- 7. ALLOCATE BONDS ---
    bonds_df = bonds_df.copy()
    bonds_df['allocation'] = bonds_budget / len(bonds_df)


    # --- 8. RETURN RESULT ---
    portfolio = []

    for _, row in etf_df.iterrows():
        portfolio.append({
            "type": "ETF",
            "name": row["name"],
            "ticker": row["ticker"],
            "allocation": round(row["allocation"], 2)
        })

    for _, row in bonds_df.iterrows():
        portfolio.append({
            "type": "Bond",
            "name": row["name"],
            "allocation": round(row["allocation"], 2)
        })

    return portfolio




In [76]:
# Call roboadvisor function
user = {
    "assetClasses": "ETFs",
    "perYearInvestment": 500,
    "riskLevel": "low",
    "selectedRegions": {"global": True, "us": True},
    "years": "3y"
}

bondDf =  pd.DataFrame({
    "name": ["US 10Y", "Germany 10Y"],
    "allocation": [5000, 5000]
})
portfolio = build_portfolio(user, df, bondDf)

for p in portfolio:
    print(p)

{'type': 'ETF', 'name': 'First Trust Global Equity Income UCITS ETF Acc', 'ticker': 'FTG4', 'allocation': 41.39}
{'type': 'ETF', 'name': 'WisdomTree Europe Equity Income UCITS ETF Acc', 'ticker': 'EEIP', 'allocation': 33.98}
{'type': 'ETF', 'name': 'Vanguard LifeStrategy 40% Equity UCITS ETF Accumulating', 'ticker': 'V40A', 'allocation': 31.3}
{'type': 'ETF', 'name': 'Vanguard LifeStrategy 60% Equity UCITS ETF Accumulating', 'ticker': 'V60A', 'allocation': 31.1}
{'type': 'ETF', 'name': 'Vanguard LifeStrategy 80% Equity UCITS ETF (EUR) Accumulating', 'ticker': 'V80A', 'allocation': 30.63}
{'type': 'ETF', 'name': 'Xtrackers Portfolio UCITS ETF 1C', 'ticker': 'XQUI', 'allocation': 30.28}
{'type': 'ETF', 'name': 'WisdomTree Emerging Markets Equity Income UCITS ETF Acc', 'ticker': 'WTD8', 'allocation': 29.14}
{'type': 'ETF', 'name': 'iShares Moderate Portfolio UCITS ETF EUR (Acc)', 'ticker': 'MODR', 'allocation': 29.08}
{'type': 'ETF', 'name': 'Invesco Global Active Defensive ESG Equity UCI